# Najafi et al. (2020) on DANDI:000016, notebook 6: previous choice and stimulus category

**Paper.** Najafi F, Elsayed GF, Cao R, Pnevmatikakis E, Latham PE, Cunningham JP, Churchland AK (2020).
Excitatory and inhibitory subnetworks are equally selective during decision-making and emerge
simultaneously during learning. *Neuron* 105, 165-179. [doi:10.1016/j.neuron.2019.09.045](https://doi.org/10.1016/j.neuron.2019.09.045)

**Dataset.** [DANDI:000016](https://dandiarchive.org/dandiset/000016/0.260915.2309), version
0.260915.2309 (doi:10.48324/dandi.000016/0.260915.2309), streamed from the archive. This is the only
published version (2026-09-15); the paper cites no DANDI version (it names the NWB deposit at the
CSHL repository, doi:10.14224/1.37693, from which the dandiset is derived), and the draft holds
the same 135 files.

**Claims.** `claims/000016/10.1016-j.neuron.2019.09.045.yaml`, written from the accepted author
manuscript in PubMed Central (PMC6952547) with its supplement, and the paper's code
(github.com/farznaj/imaging_decisionMaking_exc_inh at commit b7352acd). The claims file decides what
is tested and how; this notebook does not change a claim.

Claims tested here (the full table is in notebook 1):

| Claim | Finding | Notebook | Status |
|---|---|---|---|
| `figs6ab-previous-choice-stimulus` | Previous choice and stimulus category decoding | 06 | tested |

For run time, 5 excitatory subsamples per session are used where the claims file has 50; a
deviation, recorded.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import najafi as nj
import sessions as S

pd.set_option("display.width", 200)
MICE = [1, 2, 3, 4]
A = nj.asset_urls()
print(f"DANDI:{nj.DANDISET_ID} version {nj.DANDISET_VERSION}: {len(A)} NWB files")
items = [(i, p, u) for i, (p, u) in enumerate(zip(A.path, A.url))]

R = nj.map_sessions(S.decode_other, items, workers=4)
OK = [r for r in R if r["ok"]]
print(len(R), "sessions loaded,", len(OK), "with at least 2 inhibitory neurons")

## Data check

In [ ]:
keys = ["choice_correct", "choice_all", "init_correct", "init_all"]
d = pd.DataFrame([dict(mouse=r["mouse"], **{k: k in r["prev"] for k in keys}, stim=bool(r["stim"]),
                       cell=r.get("stim_n_min_cell", 0), dropped=r.get("stim_offset_dropped", 0)) for r in OK])
print(d.groupby("mouse").agg(sessions=("stim", "size"), **{k: (k, "sum") for k in keys}, stimulus=("stim", "sum"),
                             min_trials_per_category_and_choice=("cell", "min"), offset_trials_dropped=("dropped", "sum")).to_string())
pk = pd.Series([r["stim_bin_times"][r["stim_peak_bin"]] for r in OK if r["stim"]])
print("Bin of peak all-neuron stimulus accuracy (start, ms from stimulus onset): min %.0f, median %.0f, max %.0f" % (pk.min(), pk.median(), pk.max()))

## figs6ab-previous-choice-stimulus

> Population activity is weakly informative about the previous trial's choice and about the
> stimulus category (the stimulus category weakly except in mouse 4), and excitatory and inhibitory
> populations are similarly selective for both.

Criterion: decoders as in the opening comment of the claims file. Previous choice: readings (a)
choice-aligned, 0-97 ms before the choice, and (b) the bin before the initiation tone (the start of
the trial, before stimulus onset). Which current trials enter is not given; both (i) correct trials
(as the Figure 3 decoders) and (ii) all trials with a choice are run, and a question is asked.
Stimulus category: stimulus-aligned trials without 16 Hz, equal numbers of left and right choices in
each category; bins (a) 0-97 ms before stimulus offset and (b) the bin of peak all-neuron accuracy.
Above chance by the unpaired two-tailed t-test across sessions against the shuffle; equality by the
same test between types. Reproduced when, for both variables, the all-neuron decoder is above chance
in most mice and no mouse shows an excitatory-inhibitory difference; partially reproduced when one
mouse does.

In [ ]:
rows = []
variants = [("prev", k) for k in ("choice_correct", "choice_all", "init_correct", "init_all")] + [("stim", "offset"), ("stim", "peak")]
for var, key in variants:
    for m in MICE:
        rr = [r for r in OK if r["mouse"] == m and key in r[var]]
        v = {k: np.array([r[var][key][k] for r in rr]) for k in ("all", "all_shfl", "inh", "exc")}
        rows.append(dict(variable=var, reading=key, mouse=m, sessions=len(rr), all=v["all"].mean(), shfl=v["all_shfl"].mean(),
                         p_all_vs_shfl=nj.ttest(v["all"], v["all_shfl"])[1], exc=v["exc"].mean(), inh=v["inh"].mean(),
                         p_exc_vs_inh=nj.ttest(v["exc"], v["inh"])[1]))
T = pd.DataFrame(rows)
print(T.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
for (var, key), g in T.groupby(["variable", "reading"], sort=False):
    above = int(((g["all"] > g.shfl) & (g.p_all_vs_shfl < 0.05)).sum())
    print(f"{var} {key}: all-neuron above chance in {above} of 4 mice; mice with E-I difference: {g.mouse[g.p_exc_vs_inh < 0.05].tolist()}")

**Result: (to be written)**

## Summary

(to be written)